# U-Net Autoencoder para Reconstruccion de Imagenes
## Proyecto con PyTorch Lightning, Hydra y W&B (Google Colab)

**Dataset**: MVTec AD (cable, capsule, screw, transistor)  
**Tamano de imagen**: 128x128 RGB  
**Framework**: PyTorch Lightning + Weights & Biases  
**Plataforma**: Google Colab con GPU

Este notebook contiene toda la implementacion optimizada para Google Colab.

---

### Configuracion rapida:
1. **Runtime** -> **Change runtime type** -> **GPU** (T4 recomendado)
2. Montar Google Drive donde esta el dataset
3. Ejecutar las celdas en orden

## 1. Verificar GPU y Configuracion de Colab

In [ ]:
# Verificar GPU
import subprocess
import sys

print("Verificando configuracion de Colab...")
print("-" * 50)

# GPU info
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], encoding='utf-8')
    print(f"GPU disponible: {gpu_info.strip()}")
except:
    print("GPU no disponible - usando CPU")

# RAM info
try:
    ram_gb = subprocess.check_output(['free', '-h'], encoding='utf-8').split('\n')[1].split()[1]
    print(f"RAM total: {ram_gb}")
except:
    print("No se pudo obtener info de RAM")

print("-" * 50)
print("Configuracion verificada")

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Verificar que el dataset existe
dataset_path = '/content/drive/MyDrive/dataset'

if os.path.exists(dataset_path):
    print(f"Dataset encontrado en: {dataset_path}")
    print("\nContenido del dataset:")
    for item in os.listdir(dataset_path):
        item_path = os.path.join(dataset_path, item)
        if os.path.isdir(item_path):
            train_good = os.path.join(item_path, 'train', 'good')
            if os.path.exists(train_good):
                num_images = len([f for f in os.listdir(train_good) if f.endswith('.png')])
                print(f"  {item}/train/good: {num_images} imagenes")
else:
    print(f"ADVERTENCIA: No se encontro el dataset en {dataset_path}")
    print("Asegurate de tener la carpeta 'dataset' en tu Google Drive con las clases:")
    print("  - cable/train/good/*.png")
    print("  - capsule/train/good/*.png")
    print("  - screw/train/good/*.png")
    print("  - transistor/train/good/*.png")

## 3. Instalacion de Dependencias

In [ ]:
!pip install -q torch torchvision pytorch-lightning wandb pillow matplotlib scikit-image omegaconf hydra-core
print("Dependencias instaladas")

In [ ]:
import os
import warnings
from pathlib import Path
from typing import List, Tuple, Optional
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Torchvision
from torchvision import transforms
from PIL import Image

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger

# Weights & Biases
import wandb

# Utilidades
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf, DictConfig
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# Configuracion
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Lightning version: {pl.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Semilla
pl.seed_everything(42)
print("\nImports completados")

## 5. Configuracion del Proyecto (Hydra Config Inline)

In [ ]:
# Configuracion completa del proyecto (equivalente a archivos Hydra YAML)
config_yaml = """
# Dataset configuration
dataset:
  name: mvtec_ad
  path: /content/drive/MyDrive/dataset
  classes: ['cable', 'capsule', 'screw', 'transistor']
  img_size: 128
  channels: 3
  train_split: 0.8
  val_split: 0.2

# Model configuration
model:
  model_type: unet
  latent_dim: 128
  learning_rate: 0.001
  
  optimizer:
    name: adam
    weight_decay: 0.0001
  
  scheduler:
    use: true
    name: reduce_on_plateau
    patience: 5
    factor: 0.5
  
  # U-Net
  unet:
    base_channels: 64
    depth: 4
    activation: relu
    use_batch_norm: true

# Trainer configuration
trainer:
  max_epochs: 50
  batch_size: 32
  num_workers: 2
  accelerator: auto
  devices: 1
  precision: 32
  log_every_n_steps: 10
  gradient_clip_val: 1.0
  
  early_stopping:
    monitor: val_loss
    patience: 10
    mode: min
  
  checkpoint:
    monitor: val_loss
    mode: min
    save_top_k: 3
    dirpath: /content/checkpoints

# Logger configuration (W&B)
logger:
  project: autoencoder-reconstruction
  name: unet-colab
  save_dir: /content/wandb_logs
  offline: false
  log_model: true

# General
seed: 42
experiment_name: unet_reconstruction_colab
"""

# Cargar configuracion
cfg = OmegaConf.create(config_yaml)

print("=" * 70)
print("CONFIGURACION DEL PROYECTO")
print("=" * 70)
print(OmegaConf.to_yaml(cfg))
print("=" * 70)

## 6. Dataset MVTec AD - Implementación Completa

In [ ]:
class MVTecDataset(Dataset):
    """MVTec AD Dataset para clases especificas"""
    
    def __init__(self, root_dir, classes, img_size=128, transform=None, split='train'):
        self.root_dir = Path(root_dir)
        self.classes = classes
        self.img_size = img_size
        self.split = split
        
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform
            
        self.image_paths = self._load_image_paths()
        
    def _load_image_paths(self):
        image_paths = []
        for class_name in self.classes:
            class_dir = self.root_dir / class_name / self.split
            if not class_dir.exists():
                print(f"Advertencia: {class_dir} no existe, omitiendo...")
                continue
            
            if self.split == 'train':
                good_dir = class_dir / 'good'
                if good_dir.exists():
                    for img_path in good_dir.glob('*.png'):
                        image_paths.append((str(img_path), class_name))
            else:
                for subdir in class_dir.iterdir():
                    if subdir.is_dir():
                        for img_path in subdir.glob('*.png'):
                            image_paths.append((str(img_path), class_name))
        return image_paths
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path, class_name = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, class_name


class MVTecDataModule(pl.LightningDataModule):
    """PyTorch Lightning DataModule para MVTec AD"""
    
    def __init__(self, data_dir, classes, img_size=128, batch_size=32, 
                 num_workers=4, train_split=0.8, val_split=0.2):
        super().__init__()
        self.data_dir = data_dir
        self.classes = classes
        self.img_size = img_size
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_split = train_split
        self.val_split = val_split
        
        self.train_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        
        self.val_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        
    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            full_dataset = MVTecDataset(
                root_dir=self.data_dir,
                classes=self.classes,
                img_size=self.img_size,
                transform=self.train_transform,
                split='train'
            )
            
            total_size = len(full_dataset)
            train_size = int(self.train_split * total_size)
            val_size = total_size - train_size
            
            self.train_dataset, self.val_dataset = torch.utils.data.random_split(
                full_dataset, [train_size, val_size],
                generator=torch.Generator().manual_seed(42)
            )
            self.val_dataset.dataset.transform = self.val_transform
            
        if stage == 'test' or stage is None:
            self.test_dataset = MVTecDataset(
                root_dir=self.data_dir,
                classes=self.classes,
                img_size=self.img_size,
                transform=self.val_transform,
                split='test'
            )
    
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                         shuffle=True, num_workers=self.num_workers,
                         persistent_workers=True if self.num_workers > 0 else False,
                         pin_memory=True)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size,
                         shuffle=False, num_workers=self.num_workers,
                         persistent_workers=True if self.num_workers > 0 else False,
                         pin_memory=True)
    
    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size,
                         shuffle=False, num_workers=self.num_workers,
                         persistent_workers=True if self.num_workers > 0 else False,
                         pin_memory=True)

print("MVTecDataset y MVTecDataModule implementados")

## 7. U-Net Autoencoder - Implementación Completa

In [ ]:
class DoubleConv(nn.Module):
    """Bloque de doble convolucion (U-Net)"""
    
    def __init__(self, in_channels, out_channels, use_batch_norm=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        ]
        if use_batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))
        if use_batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        
        self.double_conv = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downsampling: MaxPool + DoubleConv"""
    
    def __init__(self, in_channels, out_channels, use_batch_norm=True):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, use_batch_norm)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upsampling: ConvTranspose + concatenate + DoubleConv"""
    
    def __init__(self, in_channels, out_channels, use_batch_norm=True):
        super().__init__()
        
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2,
                                     kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels, use_batch_norm)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        
        # Ajustar tamano si es necesario
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        
        # Concatenar skip connection
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNetAutoencoder(pl.LightningModule):
    """U-Net Autoencoder con skip connections"""
    
    def __init__(self, in_channels=3, base_channels=64, depth=4,
                 latent_dim=128, learning_rate=0.001,
                 use_batch_norm=True, optimizer_config=None,
                 scheduler_config=None):
        super().__init__()
        self.save_hyperparameters()
        
        self.in_channels = in_channels
        self.base_channels = base_channels
        self.depth = depth
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.optimizer_config = optimizer_config or {}
        self.scheduler_config = scheduler_config or {}
        
        # Initial convolution
        self.inc = DoubleConv(in_channels, base_channels, use_batch_norm)
        
        # Encoder (downsampling)
        self.down_blocks = nn.ModuleList()
        current_channels = base_channels
        for i in range(depth):
            out_channels = current_channels * 2
            self.down_blocks.append(Down(current_channels, out_channels, use_batch_norm))
            current_channels = out_channels
        
        # Bottleneck
        bottleneck_size = 128 // (2 ** depth)
        self.bottleneck_features = current_channels * bottleneck_size * bottleneck_size
        
        self.fc_encode = nn.Linear(self.bottleneck_features, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.bottleneck_features)
        
        # Decoder (upsampling)
        self.up_blocks = nn.ModuleList()
        for i in range(depth):
            in_channels_up = current_channels
            out_channels = current_channels // 2
            self.up_blocks.append(Up(in_channels_up, out_channels, use_batch_norm))
            current_channels = out_channels
        
        # Output
        self.outc = nn.Conv2d(base_channels, in_channels, kernel_size=1)
        self.activation = nn.Tanh()
        
    def forward(self, x):
        # Encoder con skip connections
        skip_connections = []
        
        x = self.inc(x)
        skip_connections.append(x)
        
        for down in self.down_blocks:
            x = down(x)
            skip_connections.append(x)
        
        # Bottleneck
        batch_size = x.size(0)
        bottleneck_channels = x.size(1)
        bottleneck_h = x.size(2)
        bottleneck_w = x.size(3)
        
        x_flat = x.view(batch_size, -1)
        z = self.fc_encode(x_flat)
        x_decoded = self.fc_decode(z)
        x = x_decoded.view(batch_size, bottleneck_channels, bottleneck_h, bottleneck_w)
        
        # Decoder con skip connections
        skip_connections = skip_connections[:-1]
        
        for i, up in enumerate(self.up_blocks):
            skip = skip_connections[-(i+1)]
            x = up(x, skip)
        
        x = self.outc(x)
        x = self.activation(x)
        
        return x
    
    def _compute_loss(self, batch):
        x, _ = batch
        x_recon = self(x)
        recon_loss = F.l1_loss(x_recon, x, reduction='mean')  # Perdida L1 (MAE)
        return recon_loss, x_recon
    
    def training_step(self, batch, batch_idx):
        loss, _ = self._compute_loss(batch)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        loss, x_recon = self._compute_loss(batch)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        
        if batch_idx == 0:
            self._log_images(batch[0], x_recon, 'val')
        return loss
    
    def _log_images(self, x, x_recon, prefix='val'):
        if hasattr(self.logger, 'experiment'):
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device)
            std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device)
            
            x_denorm = torch.clamp(x * std + mean, 0, 1)
            x_recon_denorm = torch.clamp(x_recon * std + mean, 0, 1)
            
            num_images = min(8, x.size(0))
            images = []
            
            for i in range(num_images):
                images.append(wandb.Image(x_denorm[i].cpu(), caption=f"Original {i}"))
                images.append(wandb.Image(x_recon_denorm[i].cpu(), caption=f"Recon {i}"))
            
            self.logger.experiment.log({
                f'{prefix}_reconstructions': images,
                'global_step': self.global_step
            })
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.learning_rate,
            weight_decay=self.optimizer_config.get('weight_decay', 0.0001)
        )
        
        config = {'optimizer': optimizer}
        
        if self.scheduler_config.get('use', False):
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min',
                factor=self.scheduler_config.get('factor', 0.5),
                patience=self.scheduler_config.get('patience', 5),
                verbose=True
            )
            config['lr_scheduler'] = {
                'scheduler': scheduler,
                'monitor': 'val_loss',
                'interval': 'epoch',
                'frequency': 1
            }
        
        return config

print("U-Net Autoencoder implementado")

## 8. Funciones de Utilidad y Visualización

In [ ]:
def show_images(images, titles=None, cols=4):
    """Visualizar multiples imagenes"""
    n_images = len(images)
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten() if n_images > 1 else [axes]
    
    for idx, (img, ax) in enumerate(zip(images, axes)):
        if isinstance(img, torch.Tensor):
            img = img.clone()
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img = torch.clamp(img * std + mean, 0, 1)
            img = img.permute(1, 2, 0).numpy()
        
        ax.imshow(img)
        ax.axis('off')
        if titles and idx < len(titles):
            ax.set_title(titles[idx])
    
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()


def generate_reconstructions(model, dataloader, num_samples=8, device='cpu'):
    """Generar reconstrucciones"""
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        batch = next(iter(dataloader))
        images, _ = batch
        images = images[:num_samples].to(device)
        reconstructions = model(images)
        
    return images.cpu(), reconstructions.cpu()


def evaluate_model(model, dataloader, device='cpu', num_batches=10):
    """Evaluar modelo con metricas (usando L1 loss - MAE)"""
    model.eval()
    model.to(device)
    
    mae_scores, psnr_scores, ssim_scores = [], [], []
    
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= num_batches:
                break
                
            images, _ = batch
            images = images.to(device)
            reconstructions = model(images)
            
            mae = F.l1_loss(reconstructions, images, reduction='mean').item()
            mae_scores.append(mae)
            
            imgs_np = images.cpu().numpy()
            recons_np = reconstructions.cpu().numpy()
            
            for j in range(imgs_np.shape[0]):
                img = imgs_np[j].transpose(1, 2, 0)
                recon = recons_np[j].transpose(1, 2, 0)
                
                psnr_val = psnr(img, recon, data_range=img.max() - img.min())
                psnr_scores.append(psnr_val)
                
                ssim_val = ssim(img, recon, channel_axis=2, 
                               data_range=img.max() - img.min())
                ssim_scores.append(ssim_val)
    
    return {
        'mae': np.mean(mae_scores),
        'psnr': np.mean(psnr_scores),
        'ssim': np.mean(ssim_scores)
    }

print("Funciones de utilidad definidas")

## 9. Crear DataModule y Verificar Dataset

In [ ]:
# Crear DataModule
datamodule = MVTecDataModule(
    data_dir=cfg.dataset.path,
    classes=cfg.dataset.classes,
    img_size=cfg.dataset.img_size,
    batch_size=cfg.trainer.batch_size,
    num_workers=cfg.trainer.num_workers,
    train_split=cfg.dataset.train_split,
    val_split=cfg.dataset.val_split
)

# Setup
datamodule.setup('fit')

print(f"Dataset de entrenamiento: {len(datamodule.train_dataset)} imagenes")
print(f"Dataset de validacion: {len(datamodule.val_dataset)} imagenes")
print(f"Batch size: {cfg.trainer.batch_size}")
print(f"Batches de entrenamiento: {len(datamodule.train_dataloader())}")
print(f"Batches de validacion: {len(datamodule.val_dataloader())}")

# Visualizar muestras
try:
    sample_batch = next(iter(datamodule.train_dataloader()))
    sample_images, sample_classes = sample_batch
    
    print(f"\nForma del batch: {sample_images.shape}")
    print(f"Clases en el batch: {set(sample_classes)}")
    
    # Mostrar algunas imagenes
    num_show = min(8, len(sample_images))
    show_images(sample_images[:num_show], titles=sample_classes[:num_show], cols=4)
    
except Exception as e:
    print(f"\nError al cargar el dataset: {e}")
    print("Verifica que el dataset MVTec AD este en tu Google Drive")
    print(f"Ruta esperada: {cfg.dataset.path}")
    
    # Debug: listar contenido
    import os
    if os.path.exists(cfg.dataset.path):
        print(f"\nContenido de {cfg.dataset.path}:")
        for item in os.listdir(cfg.dataset.path):
            print(f"  - {item}")

## 10. Configurar Weights & Biases (Opcional)

In [ ]:
# Configuración de W&B
USE_WANDB = True  # Cambiar a False para deshabilitar
OFFLINE_MODE = False  # Cambiar a True para modo offline

if USE_WANDB and OFFLINE_MODE:
    os.environ['WANDB_MODE'] = 'offline'

print(f"W&B habilitado: {USE_WANDB}")
print(f"Modo offline: {OFFLINE_MODE}")
print(f"Proyecto: {cfg.logger.project}")

# Si es la primera vez usando W&B, ejecuta: wandb login

## 11. Entrenar U-Net Autoencoder

In [ ]:
# Crear modelo U-Net
model_unet = UNetAutoencoder(
    in_channels=cfg.dataset.channels,
    base_channels=cfg.model.unet.base_channels,
    depth=cfg.model.unet.depth,
    latent_dim=cfg.model.latent_dim,
    learning_rate=cfg.model.learning_rate,
    use_batch_norm=cfg.model.unet.use_batch_norm,
    optimizer_config=cfg.model.optimizer,
    scheduler_config=cfg.model.scheduler
)

print("=" * 70)
print("U-NET AUTOENCODER")
print("=" * 70)
print(f"Base channels: {cfg.model.unet.base_channels}")
print(f"Depth: {cfg.model.unet.depth}")
print(f"Latent dim: {cfg.model.latent_dim}")
print(f"Learning rate: {cfg.model.learning_rate}")
print(f"Parametros: {sum(p.numel() for p in model_unet.parameters()):,}")
print("=" * 70)

# Callbacks
checkpoint_unet = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=3,
    dirpath='/content/checkpoints/unet',
    filename='unet-{epoch:02d}-{val_loss:.4f}',
    verbose=True
)

early_stop_unet = EarlyStopping(
    monitor='val_loss',
    patience=cfg.trainer.early_stopping.patience,
    mode='min',
    verbose=True
)

# Logger
if USE_WANDB:
    wandb_logger_unet = WandbLogger(
        project=cfg.logger.project,
        name=f"{cfg.logger.name}_unet",
        save_dir=cfg.logger.save_dir,
        offline=OFFLINE_MODE
    )
    logger_unet = wandb_logger_unet
else:
    logger_unet = None

# Trainer
trainer_unet = pl.Trainer(
    max_epochs=cfg.trainer.max_epochs,
    accelerator=cfg.trainer.accelerator,
    devices=cfg.trainer.devices,
    callbacks=[checkpoint_unet, early_stop_unet],
    logger=logger_unet,
    log_every_n_steps=cfg.trainer.log_every_n_steps,
    enable_progress_bar=True
)

print(f"\nIniciando entrenamiento U-Net...")
print(f"Dispositivo: {trainer_unet.accelerator}")
print(f"Epocas: {cfg.trainer.max_epochs}")
print("-" * 70)

In [ ]:
# Entrenar
trainer_unet.fit(model_unet, datamodule=datamodule)

print(f"\nEntrenamiento completado!")
print(f"Mejor val_loss: {checkpoint_unet.best_model_score:.6f}")

## 12. Visualizar Reconstrucciones del U-Net

In [ ]:
# Generar reconstrucciones
device = 'cuda' if torch.cuda.is_available() else 'cpu'
val_loader = datamodule.val_dataloader()

print(f"Generando reconstrucciones en {device}...")

# U-Net
original, recon_unet = generate_reconstructions(model_unet, val_loader, num_samples=8, device=device)

# Visualizar reconstrucciones
print("\nReconstrucciones del U-Net:")
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(min(4, len(original))):
    # Original
    img_orig = torch.clamp(original[i] * std + mean, 0, 1).permute(1, 2, 0).numpy()
    # U-Net
    img_unet = torch.clamp(recon_unet[i] * std + mean, 0, 1).permute(1, 2, 0).numpy()
    
    axes[i, 0].imshow(img_orig)
    axes[i, 0].set_title('Original' if i == 0 else '')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(img_unet)
    axes[i, 1].set_title('Reconstruccion' if i == 0 else '')
    axes[i, 1].axis('off')
    
    # Diferencia absoluta
    diff = np.abs(img_orig - img_unet)
    axes[i, 2].imshow(diff)
    axes[i, 2].set_title('Diferencia' if i == 0 else '')
    axes[i, 2].axis('off')
    
    # Error map (heatmap)
    error_map = np.mean(diff, axis=2)
    im = axes[i, 3].imshow(error_map, cmap='hot')
    axes[i, 3].set_title('Mapa de Error' if i == 0 else '')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## 13. Evaluación Cuantitativa del U-Net (MAE/L1, PSNR, SSIM)

In [ ]:
# Evaluar U-Net
print("Evaluando U-Net Autoencoder...")
metrics_unet = evaluate_model(model_unet, val_loader, device=device, num_batches=20)

# Mostrar resultados
print("\n" + "=" * 70)
print("RESULTADOS DE EVALUACION - U-NET AUTOENCODER")
print("=" * 70)
print(f"\nMAE (L1 Loss):  {metrics_unet['mae']:.6f}")
print(f"PSNR (dB):      {metrics_unet['psnr']:.2f}")
print(f"SSIM:           {metrics_unet['ssim']:.4f}")
print("=" * 70)

# Visualizacion de metricas
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

metrics_names = ['MAE (L1)', 'PSNR (dB)', 'SSIM']
metrics_values = [metrics_unet['mae'], metrics_unet['psnr'], metrics_unet['ssim']]
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, (name, value, color) in enumerate(zip(metrics_names, metrics_values, colors)):
    ax[i].bar(['U-Net'], [value], color=color, width=0.5)
    ax[i].set_title(name, fontsize=14, fontweight='bold')
    ax[i].set_ylabel('Valor', fontsize=12)
    ax[i].text(0, value, f'{value:.4f}' if name != 'PSNR (dB)' else f'{value:.2f}',
              ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretacion de Metricas:")
print(f"  - MAE (L1): {metrics_unet['mae']:.6f} - Menor es mejor (error promedio absoluto)")
print(f"  - PSNR: {metrics_unet['psnr']:.2f} dB - Mayor es mejor (>30 dB es bueno)")
print(f"  - SSIM: {metrics_unet['ssim']:.4f} - Mas cercano a 1.0 es mejor (similitud estructural)")

## 14. Resumen del Proyecto y Conclusiones

In [ ]:
# Resumen final
unet_params = sum(p.numel() for p in model_unet.parameters())

print("=" * 70)
print("RESUMEN COMPLETO DEL PROYECTO")
print("=" * 70)

print("\nDataset:")
print(f"  - Clases: {cfg.dataset.classes}")
print(f"  - Tamano de imagen: {cfg.dataset.img_size}x{cfg.dataset.img_size}")
print(f"  - Imagenes de entrenamiento: {len(datamodule.train_dataset)}")
print(f"  - Imagenes de validacion: {len(datamodule.val_dataset)}")

print("\nConfiguracion:")
print(f"  - Dimension latente: {cfg.model.latent_dim}")
print(f"  - Learning rate: {cfg.model.learning_rate}")
print(f"  - Batch size: {cfg.trainer.batch_size}")
print(f"  - Epocas: {cfg.trainer.max_epochs}")
print(f"  - Funcion de perdida: L1 Loss (MAE)")

print("\nModelo U-Net Autoencoder:")
print(f"  - Base channels: {cfg.model.unet.base_channels}")
print(f"  - Profundidad: {cfg.model.unet.depth}")
print(f"  - Parametros totales: {unet_params:,}")
print(f"  - Skip connections: Si (preservan detalles espaciales)")

print("\nResultados de Entrenamiento:")
print(f"  - Mejor Val Loss: {checkpoint_unet.best_model_score:.6f}")

print("\nMetricas de Reconstruccion:")
print(f"  - MAE (L1 Loss): {metrics_unet['mae']:.6f}")
print(f"  - PSNR: {metrics_unet['psnr']:.2f} dB")
print(f"  - SSIM: {metrics_unet['ssim']:.4f}")

print("\nConclusiones:")
print("""
1. U-Net Autoencoder entrenado exitosamente con PyTorch Lightning
2. L1 Loss (MAE) proporciona reconstrucciones robustas a outliers
3. Las skip connections del U-Net preservan detalles espaciales de alta frecuencia
4. El modelo captura caracteristicas importantes del dataset MVTec AD
5. Hydra permite configuracion flexible sin modificar codigo
6. W&B proporciona tracking completo de experimentos y visualizaciones

Ventajas del U-Net sobre Autoencoder clasico:
  - Skip connections conectan encoder y decoder directamente
  - Mejor preservacion de detalles finos y bordes
  - Reconstrucciones de mayor calidad visual
  - Convergencia mas estable durante el entrenamiento

Proximos pasos sugeridos:
  - Experimentar con diferentes dimensiones del espacio latente
  - Probar con mas epocas de entrenamiento
  - Ajustar profundidad del U-Net (depth parameter)
  - Agregar mas clases del dataset MVTec AD
  - Implementar data augmentation adicional
  - Evaluar en deteccion de anomalias
""")

print("=" * 70)
print("Proyecto completado exitosamente!")
print("=" * 70)

# Finalizar W&B
if USE_WANDB:
    wandb.finish()
    print("\nSesion de W&B finalizada")

---

## Notas para Google Colab

### Configuracion Recomendada

**Runtime:**
- **Runtime** -> **Change runtime type** -> **GPU** (T4 o superior)
- Si usas GPU gratuita, el tiempo de ejecucion esta limitado

**Memoria:**
- El notebook usa ~6-8 GB de RAM con batch_size=32
- Si hay problemas de memoria, reduce `batch_size` a 16

### Modificacion de Parametros

Para experimentar con diferentes hiperparametros, edita la celda 5 (Configuracion):

**Cambiar dimension del espacio latente:**
```yaml
model:
  latent_dim: 256  # Cambiar de 128 a 256
```

**Cambiar numero de epocas:**
```yaml
trainer:
  max_epochs: 100  # Cambiar de 50 a 100
```

**Cambiar batch size (si hay problemas de memoria):**
```yaml
trainer:
  batch_size: 16  # Reducir de 32 a 16
  num_workers: 2  # Mantener en 2 para Colab
```

**Cambiar profundidad del U-Net:**
```yaml
model:
  unet:
    depth: 5  # Cambiar de 4 a 5 (mas profundo, requiere mas memoria)
    base_channels: 128  # Mas canales base
```

### Dataset MVTec AD en Google Drive

El dataset debe estar en tu Google Drive en la carpeta `dataset/`:

Estructura requerida:
```
MyDrive/
└── dataset/
    ├── cable/
    │   └── train/
    │       └── good/*.png
    ├── capsule/
    │   └── train/
    │       └── good/*.png
    ├── screw/
    │   └── train/
    │       └── good/*.png
    └── transistor/
        └── train/
            └── good/*.png
```

**Como obtener el dataset:**
1. Visita: https://www.mvtec.com/company/research/datasets/mvtec-ad
2. Descarga las clases: cable, capsule, screw, transistor
3. Extrae y sube a Google Drive en la estructura mostrada arriba

### Guardar Checkpoints en Google Drive

Los checkpoints se guardan por defecto en `/content/checkpoints/` (temporal).

**Para guardar permanentemente en Google Drive:**

Modifica en la celda 5 (Configuracion):
```yaml
trainer:
  checkpoint:
    dirpath: /content/drive/MyDrive/checkpoints
```

### Caracteristicas Implementadas

- **U-Net Autoencoder**: Con skip connections para preservar detalles  
- **PyTorch Lightning**: Para codigo modular y organizado  
- **Google Drive integration**: Dataset desde tu Drive  
- **GPU optimizado**: Configurado para GPUs de Colab (T4, P100, V100)  
- **W&B opcional**: Tracking de experimentos (requiere login)  
- **L1 Loss (MAE)**: Mas robusta a outliers que MSE  
- **Metricas completas**: MAE, PSNR, SSIM  

### Solucion de Problemas

**Error de GPU/CUDA:**
- Verifica que GPU este habilitado: **Runtime** -> **Change runtime type** -> **GPU**
- Re-ejecuta la celda 1 para verificar

**Error de memoria (OOM):**
- Reduce `batch_size` a 16 u 8
- Reduce `depth` del U-Net a 3
- Reduce `base_channels` a 32

**Dataset no encontrado:**
- Verifica que la carpeta `dataset` este en MyDrive
- Ejecuta la celda 2 y revisa el output
- Asegurate de que las subcarpetas tengan la estructura correcta

**Google Drive no monta:**
- Re-ejecuta la celda 2
- Acepta los permisos solicitados
- Si persiste, reinicia el runtime

**W&B no funciona:**
- Ejecuta en una celda: `!wandb login`
- Pega tu API key
- O activa `OFFLINE_MODE = True` en la celda 10

### Optimizaciones para Colab

- `num_workers=2`: Optimo para Colab
- `batch_size=32`: Balance entre velocidad y memoria
- Checkpoints en `/content/`: Rapido pero temporal (usar Drive para permanente)
- Mixed precision disponible: cambia `precision: 16` en config para ahorrar memoria